# Summary of lecture 3

Yesterday we discretized a random dynamical system with additive uniform noise on the Ulam basis, we bounded the norms of the powers of the discretized annealed operator on the space of vectors of average zero, we transported those bounds from a coarse partition to a fine one, and we turned the residual of an approximate fixed point into a rigorous $L^1$ bound on the distance from the stationary density.
With that bound and a splitting of the space we enclosed the Lyapunov exponent of a family of unimodal maps at two noise sizes, one enclosure positive and the other negative.

Those norms are a mixing rate: a certified statement that after $n$ steps the operator has contracted every zero average density by a factor $\eta_n$.
Today the mixing rate is the object we go after, and the density and the Lyapunov exponent come out of it.

# A certified mixing rate

We work with the family
$$
T_{\alpha,\beta}(x) = \beta - (1+\beta)|x|^{\alpha}, \qquad \alpha \geq 1,\ \beta \in (-1, 1],
$$
on $[-1,1]$, perturbed by additive Gaussian noise of standard deviation $\sigma$, with the boundary condition that identifies $x$ with $x+2$; the technique is the one of [Galatolo, Lopez Vereau, Marangio, Nisoli, *Efficient computation of stationary measures and the Lyapunov landscape for families of random dynamical systems with smooth additive noise*](https://arxiv.org/abs/2508.03895), and the implementation is the one packaged in `PlateauExperiment.jl`.

The annealed transfer operator is again $P_\sigma f = \rho_\sigma * (Pf)$, and the reason to change basis is that convolution is multiplication frequency by frequency:
$$
\mathcal{F}(\rho_{\sigma}*f)[k] = e^{-\sigma^2 k^2 \pi^2/2}\,\mathcal{F}(f)[k].
$$
Since $P$ is a weak contraction of $L^1$, every $f \in L^1$ is sent by $P_\sigma$ to a function whose Fourier coefficients decay like $e^{-\sigma^2 k^2\pi^2/2}$; the operator is smoothing, the truncation at frequency $K$ costs an error that is exponentially small in $K^2$, and a matrix of size $2K+1$ is enough.

## Setting up

In [ ]:
import Pkg
Pkg.activate("./")
Pkg.add(["RigorousInvariantMeasures", "BallArithmetic", "IntervalArithmetic",
         "FFTW", "TaylorModels", "Plots", "RecipesBase", "LaTeXStrings"])

In [ ]:
using RigorousInvariantMeasures, IntervalArithmetic, BallArithmetic
using LinearAlgebra, FFTW, TaylorModels
using Plots, RecipesBase, LaTeXStrings

import IntervalArithmetic: inf, sup, mid, radius, diam, interval

setdisplay(:infsup; decorations = false, ng_flag = false)

We fix the parameters and build the basis. `FourierAdjoint(K, FFTNx)` is a Fourier basis truncated at the frequencies $-K, \dots, K$, which assembles the transfer operator through a discretization of its adjoint; the adjoint is the composition operator $f \mapsto f\circ T$, so the dynamic may be handed over as an ordinary function rather than as a piecewise map with its branches. The change of variables sends the system from $[-1,1]$ to $[0,1]$.

In [ ]:
α = interval(3.5)
β = interval(1)
K = 128
FFTNx = 1024

B = FourierAdjoint(K, FFTNx)

In [ ]:
τ₁(x) = (x+1)/2          # from [-1, 1] to [0, 1]
τ₂(x) = 2*x-1            # from [0, 1] to [-1, 1]

T_pm(x) = β - (1+β)*abs(x)^α
T(x) = τ₁(T_pm(τ₂(x)))

In [ ]:
plot(x -> IntervalArithmetic.mid(T(x)), 0, 1; label = "T on [0,1]")

In [ ]:
@time PK = assemble(B, T)

## Ball arithmetic

The assembled matrix is a matrix of complex intervals, and every operation we are about to perform on it is linear algebra: products, an eigendecomposition, singular values.
An interval matrix carries two endpoints per real and per imaginary part and cannot use the machine's matrix multiplication; a ball matrix carries a centre matrix and a radius matrix, so the centre product is one call to the tuned floating point routine and the radius is bounded afterwards by a few norm inequalities. We convert once, and we stay in ball arithmetic for the rest of the notebook.

In [ ]:
function ball_matrix(M)
    c = IntervalArithmetic.mid.(real.(M)) + im*IntervalArithmetic.mid.(imag.(M))
    r = sqrt.(IntervalArithmetic.radius.(real.(M)).^2 .+ IntervalArithmetic.radius.(imag.(M)).^2)
    return BallMatrix(c, r)
end

In [ ]:
bPK = ball_matrix(PK)
maximum(bPK.r)

Convolution with the noise is diagonal in this basis, with the entries written above; we build it as an interval matrix and convert it the same way.

In [ ]:
σ = interval(0.1)

NoiseInterval(σ, K) = Diagonal([[exp((-σ^2*interval(π)^2*interval(k)^2)/2) for k in 0:K];
                                [exp((-σ^2*interval(π)^2*interval(k)^2)/2) for k in -K:-1]])
bD = ball_matrix(NoiseInterval(σ, K))

In [ ]:
PσK = bD*bPK

The two heatmaps below plot $-\log|M_{ij}|$, so a light colour means an entry that is exponentially small; the second one shows what the noise does to the high frequencies. The colour scales are not the same.

In [ ]:
heatmap(-log.(abs.(BallArithmetic.mid.(bPK))); title = "deterministic")

In [ ]:
heatmap(-log.(abs.(BallArithmetic.mid.(PσK))); title = "with noise")

## The stationary density

We compute an approximate fixed point from the centre matrix, with an ordinary eigendecomposition; nothing here needs to be rigorous, since the error will be measured afterwards by the residual.

In [ ]:
F = eigen(PσK.c)
p = sortperm(abs.(F.values), rev = true)
F.values[p][1:5]

In [ ]:
scatter(F.values; label = "eigenvalues of the centre matrix")
plot!([cos(t) for t in 0:0.01:2π], [sin(t) for t in 0:0.01:2π]; label = "unit circle")

The leading eigenvalue is $1$, since the operator preserves the integral, and we normalise the corresponding eigenvector so that its zeroth coefficient is $1$.
The Fourier coefficients of a real function satisfy $\hat f[-k] = \overline{\hat f[k]}$, and the numerical eigenvector loses that symmetry; we impose it, which costs nothing and makes the density real.

In [ ]:
fσK = F.vectors[:, p[1]]
fσK /= fσK[1]

function symmetrise(v)
    w = zeros(eltype(v), length(v))
    N = (length(v)-1) ÷ 2
    w[1:N+1] = v[1:N+1]
    w[end-N+1:end] = [x' for x in reverse(v[2:N+1])]
    return w
end

fσKs = symmetrise(fσK)
bf = BallVector(fσKs)

In [ ]:
plot(real.(ifft(BallArithmetic.mid(bf))); label = "stationary density")

The residual is computed in ball arithmetic, and it is the only place where the approximate eigenvector meets a rigorous operation; $\epsilon$ below bounds $\|P_{\sigma,K} f_{\sigma,K,s} - f_{\sigma,K,s}\|_{L^2}$.

In [ ]:
res = PσK*bf - bf
ε = norm(res.c, 2) + norm(res.r, 2)

# The mixing rate

Let $V$ be the subspace of $L^2$ of average zero, which is invariant under the annealed operator; a mixing rate is a pair $(n, \eta)$ with
$$
\big\|P_{\sigma,K}^{\,n}\big|_V\big\|_{L^2\to L^2} \leq \eta < 1,
$$
proved, not observed. In the truncated basis the restriction to $V$ is the matrix with the first row and the first column removed, since the zeroth coefficient is the integral.

The eigenvalues we plotted give a candidate answer, the second largest modulus; that answer is wrong at finite $n$, and we shall see by how much.

In [ ]:
A = PσK[2:end, 2:end]
size(A)

`svd_bound_L2_opnorm` returns a rigorous upper bound for the largest singular value of a ball matrix, computed from a floating point singular value decomposition and a perturbation bound; we apply it to the powers.

In [ ]:
function power_norms(A, N)
    norms = zeros(N)
    Ai = A
    for i in 1:N
        norms[i] = BallArithmetic.svd_bound_L2_opnorm(Ai)
        Ai = Ai*A
    end
    return norms
end

In [ ]:
@time norms = power_norms(A, 12)

In [ ]:
plot(1:12, norms; yscale = :log10, marker = :circle,
     label = "certified bound for the norm of the n-th power")

In [ ]:
n₁ = findfirst(<(1.0), norms)
η  = interval(norms[n₁])
n₁, η

The first power is expansive, with norm above $1$; the second contracts, and $(n_1, \eta) = (2, 0.9193)$ is the first certified mixing rate.
Taking the twelfth power instead gives a much better rate per step, and the comparison with the second eigenvalue is the following.

In [ ]:
norms[end]^(1/12), maximum(abs.(F.values[p][2:end]))

## Why the spectrum is not the answer

The operator is not normal, so its spectrum does not control the norms of its powers at finite time; the set of points where the resolvent exceeds $1/\varepsilon$, drawn below for a coarser truncation, reaches outside the unit circle although every eigenvalue lies well inside it, which is why $\|A\|$ is above one while the spectral radius is about one half.

In [ ]:
Ksmall = 40
Bs = FourierAdjoint(Ksmall, 1024)
PKs = assemble(Bs, T)
As = (ball_matrix(NoiseInterval(σ, Ksmall))*ball_matrix(PKs)).c[2:end, 2:end]
size(As)

In [ ]:
xs = range(-1.3, 1.3; length = 90)
ys = range(-1.3, 1.3; length = 90)
@time Z = [minimum(svdvals(As - (x+im*y)*I)) for y in ys, x in xs]
contour(xs, ys, log10.(Z); levels = -6:0.25:0.5, aspect_ratio = 1,
        title = "log10 of the smallest singular value of zI - A")
plot!([cos(t) for t in 0:0.01:2π], [sin(t) for t in 0:0.01:2π]; label = "unit circle")

The figure is drawn from floating point singular values and is not certified; the numbers above it are.

# From the mixing rate to the density

We use the following statement. Let $f_\sigma$ be the fixed point of $P_\sigma$, let $f_{\sigma,K,s}$ be the symmetrized approximate fixed point computed above, let $\epsilon$ bound its residual, and suppose there are $n$ and $\eta < 1$ with $\|P_{\sigma,K}^n|_V\|_{L^2\to L^2}\leq \eta$ and constants $C_i \geq \max(1, \|P_{\sigma,K}^i|_V\|)$ for $0 \leq i \leq n-1$. Then
$$
\|f_{\sigma}-f_{\sigma,K,s}\|_{L^{2}} \leq \frac{1}{1-\eta}\sum_{i=0}^{n-1}C_i\big(\delta_{\sigma,K} + \epsilon\big),
$$
where $\delta_{\sigma,K}$ collects the error made by truncating the Fourier expansion at $K$, namely
$$
\delta_{\sigma,K} = \Gamma_{\sigma,K}(1+\Gamma^1_{\sigma,K}) + \|\rho_\sigma\|_{L^2}\sqrt{\coth(1/2\sigma^2)}\,\Gamma^1_{\sigma,K},
$$
with $\Gamma_{\sigma,K}$ the $L^1 \to L^2$ tail bound and $\Gamma^1_{\sigma,K}$ the $L^1\to L^1$ one; both carry the factor $e^{-\sigma^2K^2\pi^2/2}$.

Both $n$ and the $C_i$ come out of the computation just done, and are not free parameters: $n$ is the first index at which the certified norm drops below one, $\eta$ is the norm there, and the $C_i$ are the earlier norms, raised to $1$ where they are smaller.

In [ ]:
C = [interval(1.0); [interval(max(1.0, norms[i])) for i in 1:n₁-1]]

In [ ]:
Iπ = interval(π)

Γ  = sqrt(coth(1/(2*σ^2))/(σ*sqrt(Iπ)))*exp((-σ^2*interval(K)^2*Iπ^2)/2)
Γ¹ = (2/(σ^2*Iπ^2*interval(K)))*exp((-σ^2*interval(K)^2*Iπ^2)/2)
ρ₂ = sqrt(1/(2*σ*sqrt(Iπ)))*sqrt(coth(1/(2*σ^2)))

δ = Γ*(1+Γ¹) + ρ₂*Γ¹

At $\sigma = 0.1$ and $K = 128$ the exponent $-\sigma^2K^2\pi^2/2$ is about $-808$, so $\Gamma$ underflows the smallest positive double and the enclosure of $\delta$ is $[0, 3\cdot 10^{-323}]$; the truncation contributes nothing here, and the residual $\epsilon$ is what the bound is made of.

In [ ]:
R = (sum(C)*(δ+interval(ε)))/(1-η)

# The Lyapunov exponent

The Lyapunov exponent of the random system is
$$
\lambda(\alpha,\beta,\sigma) = \int_{-1}^{1}\log|T'_{\alpha,\beta}|\,f_{\sigma}\,dm,
$$
and, since we work with Fourier coefficients, we need those of the observable:
$$
\mathcal{F}(\log|T'|)[0] = \log((1+\beta)\alpha)-(\alpha-1),\qquad
\mathcal{F}(\log|T'|)[j] = -\frac{\alpha-1}{j\pi}\int_0^{j\pi}\frac{\sin t}{t}\,dt.
$$
We enclose the integral by splitting it at the zeros of the sine: on $[0,\pi]$ we sum the alternating power series of the primitive of $\sin(t)/t$ and bound the remainder by the first omitted term, and on each later arch $[i\pi, (i+1)\pi]$ we use a Taylor model, exactly as in lecture 2.
We work in $256$ bits, since the power series at $0$ has terms with factorials in the denominator.

In [ ]:
setprecision(256)
Pi = interval(BigFloat, π)

sinc_over(t) = sin(t)/t
tay(i, x) = x^(2i+1)/(interval(BigFloat, factorial(big(2i+1)))*interval(BigFloat, 2i+1))

In [ ]:
Nser = 60
I₀ = sum([(-1)^i*tay(i, Pi) for i in 0:Nser]) + interval(BigFloat, -1, 1)*abs(tay(Nser+1, Pi))

In [ ]:
function int_over_arch(f, i; degree = 40)
    I = interval(inf(interval(BigFloat, i)*Pi), sup(interval(BigFloat, i+1)*Pi))
    m = interval(BigFloat, mid(I))
    prim = TaylorSeries.integrate(f(TaylorModel1(degree, m, I)))
    return prim(sup(I)-m) - prim(inf(I)-m)
end

In [ ]:
@time arcs = [I₀; [int_over_arch(sinc_over, i) for i in 1:K-1]]
maximum(diam.(arcs))

The cumulative sum of the arches gives $\int_0^{j\pi}\sin(t)/t\,dt$; dividing by $j\pi$ gives the coefficients. The alternating sign that follows converts the coefficients on $[-1,1]$ into the coefficients on $[0,1]$, which is the coordinate change we made at the beginning.

In [ ]:
coeff = cumsum(arcs) ./ [interval(BigFloat, i)*Pi for i in 1:K]
coeff01 = [(-1)^i for i in 1:K] .* coeff

αb = interval(BigFloat, 3.5)
βb = interval(BigFloat, 1)
lnn = [log((1+βb)*αb) - (αb-1); -(αb-1)*[coeff01; reverse(coeff01)]]
length(lnn), lnn[1], lnn[2]

In [ ]:
λK = sum(lnn[i]*(interval(BigFloat, real(fσKs[i])) + im*interval(BigFloat, imag(fσKs[i])))
         for i in 1:2K+1)
real(λK)

The last ingredient turns the $L^2$ error on the density into an error on the integral: by Cauchy-Schwarz,
$$
|\lambda(\alpha,\beta,\sigma) - \langle \log|T'|, f_{\sigma,K,s}\rangle| \leq \Upsilon\,\|f_\sigma - f_{\sigma,K,s}\|_{L^2},
\qquad \Upsilon = \|\log|T'|\|_{L^2},
$$
and for this family $\Upsilon$ is known in closed form,
$$
\Upsilon(\alpha,\beta) = \sqrt{2}\Big(\big(\log((1+\beta)\alpha)-(\alpha-1)\big)^2+(\alpha-1)^2\Big)^{1/2}.
$$

In [ ]:
Υ = sqrt(interval(BigFloat, 2))*((log((βb+1)*αb)-(αb-1))^2 + (αb-1)^2)^interval(BigFloat, 0.5)

In [ ]:
Rb = interval(BigFloat, inf(R), sup(R))
λ = real(λK) + Υ*Rb*interval(BigFloat, -1, 1)

In [ ]:
diam(λ)

The enclosure is about $2\cdot 10^{-10}$ wide, and it is positive: at $\alpha = 3.5$, $\beta = 1$ and $\sigma = 0.1$ the orbits of this random system separate.
Every link in the chain came from a certified number: the matrix from interval arithmetic, the mixing rate from certified singular values, the residual and the density error from ball arithmetic, the observable from Taylor models.

# The Gauss map in a Chebyshev basis

The same three steps, discretize, enclose, certify, work in any basis in which the operator is close to a matrix; we close with the Gauss map $x \mapsto 1/x \bmod 1$, whose transfer operator is
$$
(Lf)(x) = \sum_{n\geq 1}\frac{1}{(n+x)^2}f\Big(\frac{1}{n+x}\Big),
$$
the Gauss-Kuzmin-Wirsing operator. It has infinitely many branches, so the sum has to be truncated and its tail enclosed; on the Chebyshev basis of $[0,1]$ the coefficients of an analytic function decay geometrically, and a matrix of size $21$ is enough for several digits.

We sum the first $N$ branches term by term, and for the rest we use that $T_k(2u-1) = (-1)^k + O(k^2 u)$ as $u\to 0$: the leading part is $(-1)^k\sum_{n>N}(n+x)^{-2}$, which the integral comparison encloses between $1/(N+1+x)$ and $1/(N+x)$, and the remainder is bounded by $2k^2\sum_{n>N}(n+x)^{-3} \leq k^2/(N+x)^2$.

In [ ]:
KC = 20
NB = 20_000
nodes = [(1+cos(interval(π)*interval(j)/interval(KC)))/2 for j in 0:KC]

In [ ]:
function chebvec(u, K)                     # T_0(u), ..., T_K(u) by the recurrence
    t = Vector{typeof(u)}(undef, K+1)
    t[1] = one(u)
    t[2] = u
    for m in 2:K
        t[m+1] = 2*u*t[m] - t[m-1]
    end
    return t
end

function gkw_values(nodes, KC, NB)
    V = zeros(Interval{Float64}, KC+1, KC+1)
    for (j, x) in enumerate(nodes)
        acc = zeros(Interval{Float64}, KC+1)
        for m in 1:NB
            y = interval(m) + x
            acc .+= chebvec(2/y-1, KC) ./ y^2
        end
        S₂ = interval(inf(1/(interval(NB+1)+x)), sup(1/(interval(NB)+x)))
        S₃ = interval(0, sup(1/(2*(interval(NB)+x)^2)))
        for k in 0:KC
            acc[k+1] += (-1)^k*S₂ + interval(-1, 1)*2*interval(k)^2*S₃
        end
        V[j, :] = acc
    end
    return V
end

In [ ]:
@time V = gkw_values(nodes, KC, NB)
size(V)

The values at the Chebyshev points are turned into Chebyshev coefficients by the discrete cosine transform, written here as a matrix so that the whole step stays inside interval arithmetic.

In [ ]:
function cheb_coefficient_matrix(KC)
    W = zeros(Interval{Float64}, KC+1, KC+1)
    for m in 0:KC, j in 0:KC
        c = (j == 0 || j == KC) ? interval(0.5) : interval(1.0)
        W[m+1, j+1] = 2*c*cos(interval(π)*interval(m)*interval(j)/interval(KC))/interval(KC)
    end
    W[1, :] ./= 2
    W[end, :] ./= 2
    return W
end

M = cheb_coefficient_matrix(KC)*V
maximum(radius.(M))

In [ ]:
bM = BallMatrix(IntervalArithmetic.mid.(M), IntervalArithmetic.radius.(M))
FG = eigen(bM.c)
q = sortperm(abs.(FG.values), rev = true)
FG.values[q][1:5]

`certify_eigenpair` runs a Newton-Kantorovich test on the augmented system that has the eigenpair as its root: it needs an approximate eigenvalue and eigenvector, and it returns either a radius within which a true eigenpair of the ball matrix is proved to lie, or a failure.

In [ ]:
for i in q[1:3]
    r = certify_eigenpair(bM, FG.values[i], FG.vectors[:, i])
    println((r.verified, r.eigenvalue, r.enclosure_radius))
end

The first eigenvalue is enclosed around $1$, which is a check on the discretization rather than a result, since the operator preserves the integral.
The second is the Gauss-Kuzmin-Wirsing constant, enclosed within about $2.3\cdot 10^{-5}$ of $-0.303663$; the third fails the test at this truncation, the Newton-Kantorovich condition not being satisfied.

We remark that what has been certified is an eigenvalue of the $21\times 21$ ball matrix, not of the operator; the step from one to the other needs a bound on the resolvent along a contour separating the eigenvalue, and it is carried out in `GKWExperiments.jl`, where the matrix entries are built from Hurwitz zeta values instead of a truncated branch sum and the tail no longer limits the accuracy.

# Summary of the lecture

We discretized the annealed transfer operator of a random system with Gaussian noise on a Fourier basis, where the noise is diagonal and the truncation error is exponentially small in $K^2$, and we worked throughout in ball arithmetic.

From certified singular values of the powers of the operator restricted to the zero average subspace we obtained a mixing rate, a proved pair $(n, \eta)$ with $\|P^n|_V\| \leq \eta < 1$; the first power has norm above one although the spectral radius is about one half, which is what non-normality does and what the pseudospectrum picture shows.

The mixing rate, the residual of an approximate fixed point and the truncation constants gave an $L^2$ bound on the distance between the computed density and the true one, and the Fourier coefficients of $\log|T'|$, enclosed with Taylor models, turned that into an enclosure of the Lyapunov exponent.

The closing section applied the same three steps in a Chebyshev basis to the Gauss-Kuzmin-Wirsing operator, and certified its second eigenvalue.